## Tabela: `workspace.silver.meta_alfabetizacao_brasil`

| Item | Detalhe |
|------|---------|
| **Tabela** | `workspace.silver.meta_alfabetizacao_brasil` |
| **Origem** | `workspace.bronze.meta_alfabetizacao_brasil_raw` |
| **Descrição** | Tratamento e padronização dos dados de metas de alfabetização por rede de ensino, previamente ingeridos na camada Bronze. |
| **Camada** | Silver (tratamento e padronização) |
| **Colunas de controle** | `data_ingestao` (timestamp de carga), `fonte` (origem dos dados) |
| **Criado por** | suelen.ramos |
| **Data de criação** | 2025-01-20 |

### Chave de negócio
A combinação `ano + rede_codigo` forma a chave única da tabela.

### Tratamentos realizados

- Padronização dos tipos de dados;
- Remoção de espaços da coluna `rede`;
- Padronização do nome da rede para `rede_codigo`;
- Preservação das metas por ano;
- Preservação das colunas de controle da ingestão;
- Validação de duplicidades após a transformação.

In [0]:
%run ../../Monitoring/_common_monitoring

In [0]:
_m = iniciar_etapa()

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.silver.meta_alfabetizacao_brasil
USING DELTA
AS
SELECT
    CAST(ano AS INT) AS ano,
    TRIM(CAST(rede AS STRING)) AS rede_codigo,

    CAST(taxa_alfabetizacao AS DOUBLE) AS taxa_alfabetizacao,

    CAST(meta_alfabetizacao_2024 AS DOUBLE) AS meta_alfabetizacao_2024,
    CAST(meta_alfabetizacao_2025 AS DOUBLE) AS meta_alfabetizacao_2025,
    CAST(meta_alfabetizacao_2026 AS DOUBLE) AS meta_alfabetizacao_2026,
    CAST(meta_alfabetizacao_2027 AS DOUBLE) AS meta_alfabetizacao_2027,
    CAST(meta_alfabetizacao_2028 AS DOUBLE) AS meta_alfabetizacao_2028,
    CAST(meta_alfabetizacao_2029 AS DOUBLE) AS meta_alfabetizacao_2029,
    CAST(meta_alfabetizacao_2030 AS DOUBLE) AS meta_alfabetizacao_2030,

    CAST(percentual_participacao AS DOUBLE) AS percentual_participacao,

    data_ingestao,
    fonte,
    lote_id

FROM workspace.bronze.meta_alfabetizacao_brasil_raw

WHERE ano IS NOT NULL
  AND rede IS NOT NULL

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY ano, rede
    ORDER BY data_ingestao DESC
) = 1;


In [0]:
registrar_execucao("SILVER", "meta_brasil", _m, row_count=contar_tabela("workspace.silver.meta_alfabetizacao_brasil"))